# Learning Objectives — What Is the Model Actually Trained to Do?

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q torch numpy matplotlib

## Why the training objective is everything

You now understand how a Transformer is built — the architecture, the attention mechanism, all the moving parts.

But knowing the architecture is like knowing how a brain is wired. It does not tell you what the brain has learned to do. A surgeon and a poet have almost identical brains — what makes them different is what each one practiced over years.

For a model, **the training objective determines what it practices**. It is the task the model is asked to do millions of times until it gets good at it. The architecture is just the tool — the objective shapes the behavior.

This is why:
- GPT can write an essay but is not naturally good at answering questions about a document
- BERT is excellent at understanding text but cannot generate new sentences on its own
- An embedding model clusters similar sentences together but does not try to predict the next word

Same basic architecture. Completely different objectives. Completely different capabilities.

In this notebook we will cover three training objectives that power most of modern GenAI:

1. **Next Token Prediction** — the objective behind GPT, LLaMA, Claude
2. **Masked Language Modelling** — the objective behind BERT
3. **Contrastive Learning** — the objective behind embedding models like CLIP and Sentence-BERT

## 1. Next Token Prediction — the autocomplete that became a genius

The training objective for GPT, LLaMA, and most large language models is almost embarrassingly simple.

Given the words so far, predict the next one.

That is it.

**The analogy:** Remember the days of T9 texting? You press a few keys and your old Nokia suggests the next word. Next token prediction is that — except instead of a dictionary lookup, the model reads every book, article, forum post, and code repository ever written, and learns to predict what comes next from the full context of everything that came before.

The surprising thing is that to predict the next word well, the model is forced to learn an enormous amount:

- Grammar — because grammatically wrong continuations are unlikely
- Facts — because factually wrong continuations are unlikely
- Reasoning — because illogical continuations are unlikely
- Style and tone — because out-of-character continuations are unlikely

Nobody tells the model to learn any of this. It all falls out of trying to predict the next word, over and over, across billions of examples.

**The training process:** At each step, the model sees a sequence of tokens and predicts the probability of each possible next token. It compares its prediction to the actual next token. If it was wrong, the weights are nudged slightly to make the correct token more likely next time. Repeat a trillion times.

In [ ]:
import torch
import torch.nn.functional as F

# A tiny vocabulary for illustration
vocab = ["The", "cat", "sat", "on", "the", "mat", "ran", "away", "<unk>"]
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for i, w in enumerate(vocab)}

# Our training sentence: "The cat sat on the mat"
sentence = ["The", "cat", "sat", "on", "the", "mat"]
ids = [word_to_id[w] for w in sentence]

print("Training pairs generated from: 'The cat sat on the mat'\n")
print(f"{'Input (context)':<30}  {'Target (correct next token)'}")
print("-" * 60)

for i in range(1, len(ids)):
    context = " ".join(sentence[:i])
    target  = sentence[i]
    print(f"  {context:<28}  →  '{target}'")

print()
print("Every sentence automatically generates training pairs — no human labelling needed.")
print("This is why LLMs can be trained on the entire internet. The data labels itself.")

In [ ]:
import torch
import torch.nn.functional as F

# Simulate the model outputting raw scores (logits) for each vocab word
# In a real model these come from the final linear layer
torch.manual_seed(7)
vocab_size = len(vocab)
logits = torch.randn(vocab_size)   # random scores at the start of training

# The correct next word after "The cat" is "sat" (index 2)
context       = "The cat"
correct_word  = "sat"
correct_index = torch.tensor(word_to_id[correct_word])

# Cross-entropy loss — measures how wrong the prediction is
loss = F.cross_entropy(logits.unsqueeze(0), correct_index.unsqueeze(0))

probs = F.softmax(logits, dim=0)

print(f"Context: '{context}'  →  correct next word: '{correct_word}'\n")
print(f"{'Word':<10}  {'Probability':>12}  {'<-- correct' if False else ''}")
print("-" * 35)
for i, (word, prob) in enumerate(zip(vocab, probs)):
    marker = "  ← correct" if word == correct_word else ""
    print(f"  {word:<10}  {prob.item():>10.4f}{marker}")

print(f"\nLoss: {loss.item():.4f}")
print("A high loss means the model assigned low probability to the correct word.")
print("Training nudges the weights until 'sat' gets a much higher probability.")

## 2. Masked Language Modelling — the fill-in-the-blank exam

BERT (and models like RoBERTa) use a different training objective. Instead of predicting what comes next, they randomly hide some words in the middle of a sentence and ask the model to fill them back in.

**The analogy:** Imagine a fill-in-the-blank exam from school.

*"The ___ barked loudly at the stranger."*

You do not just look left to right — you read the whole sentence, including the words after the blank, to figure out the answer is probably "dog". This is the key difference from next token prediction: BERT reads context from **both directions** simultaneously.

About 15% of tokens are randomly masked during training. The model has to predict what was hidden using all the surrounding context — left and right. This forces the model to build a very deep understanding of how words relate to each other in both directions.

**What this is good for:** Because BERT reads everything at once, it is excellent at understanding tasks — classifying sentiment, finding named entities, answering questions given a passage. It cannot generate new text (it was never trained to), but it understands existing text better than a pure left-to-right model.

In [ ]:
import random

random.seed(42)

def mask_sentence(sentence, mask_prob=0.15, mask_token="[MASK]"):
    words  = sentence.split()
    masked = []
    labels = []   # what the model needs to predict

    for word in words:
        if random.random() < mask_prob:
            masked.append(mask_token)
            labels.append(word)    # the correct answer
        else:
            masked.append(word)
            labels.append(None)   # not masked, nothing to predict

    return masked, labels

sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Transformers have completely changed the field of natural language processing",
    "The stock market fell sharply after the central bank raised interest rates",
]

print("Masked Language Modelling — training examples\n")
for sentence in sentences:
    masked, labels = mask_sentence(sentence, mask_prob=0.25)
    masked_str = " ".join(masked)
    targets    = [(m, l) for m, l in zip(masked, labels) if l is not None]

    print(f"Original : {sentence}")
    print(f"Masked   : {masked_str}")
    if targets:
        print(f"Targets  : {targets}")
    else:
        print("Targets  : (nothing masked this time — try running again)")
    print()

print("The model sees the masked sentence and must predict the hidden words.")
print("It uses context from both the left AND the right of each [MASK].")

## 3. Contrastive Learning — pulling friends together, pushing strangers apart

Embedding models — the ones that power semantic search and RAG systems — are trained using a completely different approach called **contrastive learning**.

The goal is simple: similar things should have similar embeddings, and different things should have different embeddings.

**The analogy:** Imagine you are organising a party and you have two tasks. First, you want to seat people with similar interests at the same table — pull them together. Second, you want to make sure people with nothing in common are not awkwardly seated next to each other — push them apart.

Contrastive learning trains a model by showing it pairs:
- **Positive pairs** — two sentences that mean similar things. The model is penalised if their embeddings are far apart.
- **Negative pairs** — two sentences with unrelated meanings. The model is penalised if their embeddings are close together.

Over millions of examples, the model learns to arrange meaning in space — similar meanings cluster together, different meanings spread apart.

**The training signal — InfoNCE loss:** For each anchor sentence, there is one correct positive match and many negatives (often from the same batch). The model must assign high similarity to the positive and low similarity to all negatives. This is exactly the same challenge as softmax — identify the one correct answer from a list of options.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def softmax(x):
    x = x - np.max(x)
    return np.exp(x) / np.exp(x).sum()

# Imagine these are embeddings produced by a model
# Positive pair: two sentences about the same topic
# Negatives: sentences from different topics in the same batch

np.random.seed(5)
d = 8

# Anchor: "The dog ran across the park"
anchor   = np.random.randn(d)

# Positive: "A dog was running in the garden" (same meaning)
positive = anchor + np.random.randn(d) * 0.3   # close to anchor

# Negatives: unrelated sentences in the same batch
negatives = [
    np.random.randn(d),   # "The stock market closed higher today"
    np.random.randn(d),   # "She ordered a large coffee with oat milk"
    np.random.randn(d),   # "The spacecraft completed its third orbit"
]

neg_labels = [
    "The stock market closed higher today",
    "She ordered a large coffee with oat milk",
    "The spacecraft completed its third orbit",
]

# Similarity scores
sim_positive  = cosine_similarity(anchor, positive)
sim_negatives = [cosine_similarity(anchor, n) for n in negatives]

all_sims = np.array([sim_positive] + sim_negatives)
probs    = softmax(all_sims / 0.07)   # temperature scaling — sharper distribution

print("Anchor: 'The dog ran across the park'\n")
print(f"{'Sentence':<45}  {'Similarity':>10}  {'Prob after softmax':>18}")
print("-" * 80)

all_sentences = ["A dog was running in the garden  ← POSITIVE"] + neg_labels
for sentence, sim, prob in zip(all_sentences, all_sims, probs):
    print(f"  {sentence:<43}  {sim:>10.4f}  {prob:>17.4f}")

print()
# InfoNCE loss = -log(prob of the positive)
loss = -np.log(probs[0] + 1e-9)
print(f"InfoNCE loss: {loss:.4f}")
print()
print("A low loss means the model assigned high probability to the positive pair.")
print("Training pushes the model to cluster similar meanings and separate different ones.")

## Seeing contrastive learning work — before and after training

Let us simulate what embeddings look like before training (random, no structure) and after contrastive training (similar meanings cluster together).

In a real model, this structure emerges after millions of gradient updates. Here we simulate it with a small tweak to the embeddings to illustrate the idea.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(99)

groups = {
    "Food": [
        "I love spicy biryani",
        "Pizza is the best comfort food",
        "She cooked a delicious curry",
    ],
    "Sports": [
        "The team scored in the final minute",
        "He trains every morning for the marathon",
        "Cricket is popular across South Asia",
    ],
    "Technology": [
        "The new chip is twice as fast",
        "Open source software changed the industry",
        "She wrote the backend in Python",
    ],
}
colors = {"Food": "tomato", "Sports": "steelblue", "Technology": "seagreen"}
d = 16

# Before training — random embeddings, no structure
before_embs = {g: np.random.randn(3, d) for g in groups}

# After contrastive training — same-group embeddings are pulled together
after_embs = {}
group_centers = {g: np.random.randn(d) * 2 for g in groups}
for g, center in group_centers.items():
    after_embs[g] = center + np.random.randn(3, d) * 0.3   # tight cluster

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, emb_dict, title in zip(
    axes,
    [before_embs, after_embs],
    ["Before training — random, no structure", "After contrastive training — similar meanings cluster"]
):
    all_embs   = np.vstack([e for e in emb_dict.values()])
    all_labels = [g for g, e in emb_dict.items() for _ in range(len(e))]
    all_texts  = [s for sentences in groups.values() for s in sentences]

    reduced = PCA(n_components=2).fit_transform(all_embs)

    for i, (label, text) in enumerate(zip(all_labels, all_texts)):
        ax.scatter(reduced[i, 0], reduced[i, 1], color=colors[label], s=120, zorder=3)
        short = text[:22] + "..." if len(text) > 22 else text
        ax.annotate(short, (reduced[i, 0], reduced[i, 1]),
                    textcoords="offset points", xytext=(6, 4), fontsize=7)

    for group, color in colors.items():
        ax.scatter([], [], color=color, label=group, s=80)

    ax.legend(fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PCA dim 1")
    ax.set_ylabel("PCA dim 2")
    ax.grid(True, alpha=0.2)

plt.suptitle("What contrastive learning does to the embedding space", fontsize=12)
plt.tight_layout()
plt.show()

print("Left:  before training — food, sports and tech sentences are scattered randomly.")
print("Right: after training  — each topic forms a tight cluster, topics stay far apart.")
print("This is the structure that makes semantic search work.")

## How the same architecture becomes three different models

Here is the part that surprises most people. GPT, BERT, and a sentence embedding model all use a Transformer architecture with attention, feed-forward layers, residual connections — basically the same building blocks you saw in the previous notebook.

What makes them completely different is the training objective.

| | GPT / LLaMA / Claude | BERT / RoBERTa | Sentence-BERT / CLIP |
|---|---|---|---|
| **Objective** | Predict the next token | Predict masked tokens | Pull similar pairs together |
| **Reads context** | Left to right only | Both directions | Both directions |
| **Output** | Next token probabilities | Filled-in token | Dense embedding vector |
| **Good at** | Generating text | Understanding text | Semantic similarity & search |
| **Cannot** | See future tokens during training | Generate freely | Produce natural language |

**The deeper point:** the objective does not just change what the model can do. It changes what the model internalises during training. A next-token model implicitly learns causality and narrative flow because those matter for predicting what comes next. A masked model learns bidirectional syntax and coreference because those help fill in blanks. A contrastive model learns semantic proximity because that is literally what it is rewarded for.

You cannot train a model on next-token prediction and expect it to give great embeddings for semantic search. The objective shapes the internal representations, not just the outputs.

In [ ]:
# A simple summary table — no model needed, just clear thinking

objectives = [
    {
        "name"      : "Next Token Prediction",
        "models"    : "GPT-4, LLaMA 3, Claude, Gemini",
        "question"  : "What word comes next?",
        "example"   : "'The cat sat on the' → 'mat'",
        "best_for"  : "Text generation, chat, code",
    },
    {
        "name"      : "Masked Language Modelling",
        "models"    : "BERT, RoBERTa, DeBERTa",
        "question"  : "What word was hidden here?",
        "example"   : "'The [MASK] sat on the mat' → 'cat'",
        "best_for"  : "Classification, NER, question answering",
    },
    {
        "name"      : "Contrastive Learning",
        "models"    : "Sentence-BERT, CLIP, text-embedding-3",
        "question"  : "Are these two sentences similar?",
        "example"   : "('Dog runs in park', 'A dog was running') → similar",
        "best_for"  : "Semantic search, RAG retrieval, clustering",
    },
]

for obj in objectives:
    print(f"Objective : {obj['name']}")
    print(f"Models    : {obj['models']}")
    print(f"Question  : {obj['question']}")
    print(f"Example   : {obj['example']}")
    print(f"Best for  : {obj['best_for']}")
    print()

## Key takeaways

- The **training objective** is the task the model practices millions of times. It shapes everything — what the model learns, what it is good at, and what it cannot do.
- **Next token prediction** (GPT, LLaMA, Claude) — predict the next word given everything before. Deceptively simple, but forces the model to learn grammar, facts, reasoning, and style just to do it well.
- **Masked language modelling** (BERT) — predict randomly hidden words using context from both directions. Builds deep bidirectional understanding. Cannot generate freely.
- **Contrastive learning** (Sentence-BERT, CLIP) — pull similar pairs together and push unrelated pairs apart. Produces embeddings where meaning lives in proximity. Powers semantic search and RAG.
- The same Transformer architecture, trained on different objectives, produces models with completely different strengths. Choosing the right model for a task starts with asking: what was it trained to do?

---

Next up: **Training Mechanics** — now that you know what the model is trained to do, the next question is how it actually learns. Optimisers, learning rate schedules, and the tricks that make training a 70-billion-parameter model not fall apart.